# 0. Retrain Model Inside Container

Run this notebook **once** before running notebooks 1 and 2.

It regenerates the training data and retrains the Random Forest classifier
using the scikit-learn version installed in this container, so the model
is fully compatible with the consumer classifier.

The retrained model overwrites `models/classifier.joblib`.

In [1]:
import sys, os, random, re, string, urllib.parse
import numpy as np
import pandas as pd
import sklearn

sys.path.insert(0, '/home/jovyan/work')
from src.classifier import AttackClassifier

print(f'scikit-learn version: {sklearn.__version__}')
print('Generating synthetic training data...')

scikit-learn version: 1.8.0
Generating synthetic training data...


In [2]:
# ── Load real CSIC 2010 data ───────────────────────────────────────────────
# Using the actual dataset gives much more diverse patterns and realistic
# confidence scores compared to training on the same synthetic data the
# producer uses.
sys.path.insert(0, '/home/jovyan/work')
from src.preprocessor import load_dataset

import pathlib

# The compose.yml mounts  ../data  →  /home/jovyan/work/data
csv_abs = '/home/jovyan/work/data/csic_database.csv'
print(f'Looking for dataset at: {csv_abs}')

if not pathlib.Path(csv_abs).exists():
    # Fallback: generate synthetic data if CSIC CSV not found
    print('CSIC 2010 CSV not found — falling back to synthetic data.')
    print('For better results mount the data/ folder and rerun.')
    USE_SYNTHETIC = True
else:
    print(f'Found CSIC 2010 dataset. Loading...')
    USE_SYNTHETIC = False


Looking for dataset at: /home/jovyan/work/data/csic_database.csv
Found CSIC 2010 dataset. Loading...


In [3]:
# ── Train on CSIC 2010 only ─────────────────────────────────────────────────
# No synthetic data — the producer generates novel payloads the model has
# never seen, which gives realistic (non-100%) confidence scores.
if USE_SYNTHETIC:
    raise RuntimeError(
        'CSIC 2010 dataset not found.\n'
        'Mount the data/ folder and rerun:\n'
        '  volume: ../data:/home/jovyan/work/data'
    )

df = load_dataset('/home/jovyan/work/data')
requests = df.to_dict('records')
labels   = df['label'].tolist()
print(f'Training on {len(requests)} real CSIC 2010 requests  '
      f'({labels.count("normal")} normal / {labels.count("anomalous")} anomalous)')

clf = AttackClassifier()
clf.train(requests, labels, verbose=True)
clf.save('/home/jovyan/work/models/classifier.joblib')
print('\nModel retrained and saved.')


  Loading /home/jovyan/work/data/csic_database.csv ...
  Loaded 8000 requests — 5000 normal, 3000 anomalous
Training on 8000 real CSIC 2010 requests  (5000 normal / 3000 anomalous)

Training classification report:
               precision    recall  f1-score   support

   anomalous       1.00      1.00      1.00      3000
      normal       1.00      1.00      1.00      5000

    accuracy                           1.00      8000
   macro avg       1.00      1.00      1.00      8000
weighted avg       1.00      1.00      1.00      8000

Model saved to /home/jovyan/work/models/classifier.joblib

Model retrained and saved.


In [4]:
# Debug: inspect features for key test cases
import sys, importlib
sys.path.insert(0, '/home/jovyan/work')
import src.features as feat_mod
importlib.reload(feat_mod)
from src.features import extract_features
import pandas as pd

debug_cases = [
    ('normal_anadir',    {'method':'GET',  'url':'/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=150&cantidad=2',  'body':'', 'headers':{}}),
    ('normal_login',     {'method':'POST', 'url':'/tienda1/publico/autenticar.jsp', 'body':'correo=juan@mail.com&password=Pass1234', 'headers':{}}),
    ('attack_sqli',      {'method':'POST', 'url':'/tienda1/publico/autenticar.jsp', 'body':"correo=' OR '1'='1'--&password=x", 'headers':{}}),
    ('attack_traversal', {'method':'GET',  'url':'/tienda1/publico/caracteristicas.jsp?id=../../../etc/passwd', 'body':'', 'headers':{}}),
    ('attack_tampering', {'method':'GET',  'url':'/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=-1&cantidad=-99999', 'body':'', 'headers':{}}),
]

rows = []
for label, req in debug_cases:
    f = extract_features(req)
    f['_case'] = label
    rows.append(f)

df = pd.DataFrame(rows).set_index('_case')
varying = df.columns[df.nunique() > 1]
print(df[varying].to_string())
print()
print('Total features:', len(df.columns))
if 'has_negative_param' in df.columns:
    print('has_negative_param:', df['has_negative_param'].to_dict())
else:
    print('WARNING: has_negative_param MISSING — features.py not updated in container!')


                  url_length  body_length  num_params  url_special_chars  body_special_chars  has_sql_operator  has_cmd_injection  has_path_traversal  is_post  is_get  has_negative_param
_case                                                                                                                                                                                     
normal_anadir             72            0           4                  3                   0                 0                  0                   0        0       1                   0
normal_login              31           38           0                  0                   1                 0                  0                   0        1       0                   0
attack_sqli               31           32           0                  0                   6                 1                  0                   0        1       0                   0
attack_traversal          59            0           1            

In [5]:
# -- Verify ─────────────────────────────────────────────────────────────────
clf2 = AttackClassifier.load('/home/jovyan/work/models/classifier.joblib')

# URLs match CSIC 2010 format exactly (id + nombre + precio + cantidad)
tests = [
    ({'method':'GET',
      'url':'/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=150&cantidad=2',
      'body':'', 'headers':{}}, 'normal'),
    ({'method':'GET',
      'url':'/tienda1/publico/buscar.jsp?id=5&nombre=Aceite+Oliva&precio=200&cantidad=1',
      'body':'', 'headers':{}}, 'normal'),
    ({'method':'POST',
      'url':'/tienda1/publico/autenticar.jsp',
      'body':'correo=juan@mail.com&password=Pass1234',
      'headers':{}}, 'normal'),
    ({'method':'POST',
      'url':'/tienda1/publico/autenticar.jsp',
      'body':"correo=' OR '1'='1'--&password=x",
      'headers':{}}, 'anomalous'),
    ({'method':'GET',
      'url':'/tienda1/publico/buscar.jsp?query=%3Cscript%3Ealert(1)%3C%2Fscript%3E',
      'body':'', 'headers':{}}, 'anomalous'),
    ({'method':'GET',
      'url':'/tienda1/publico/caracteristicas.jsp?id=../../../etc/passwd',
      'body':'', 'headers':{}}, 'anomalous'),
]

print('Verification:')
all_ok = True
for req, expected in tests:
    r    = clf2.predict(req)
    ok   = r['label'] == expected
    all_ok = all_ok and ok
    icon = '\u2705' if ok else '\u274c'
    print(f'  {icon} expected={expected:<10} got={r["label"]:<10} conf={r["confidence"]:.0%}  {req["url"][:60]}')

print()
print('\u2705 Model ready \u2014 run notebooks 1 and 2.' if all_ok else '\u274c Some checks failed \u2014 check training data.')


Verification:
  ✅ expected=normal     got=normal     conf=97%  /tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=15
  ✅ expected=normal     got=normal     conf=100%  /tienda1/publico/buscar.jsp?id=5&nombre=Aceite+Oliva&precio=
  ✅ expected=normal     got=normal     conf=100%  /tienda1/publico/autenticar.jsp
  ✅ expected=anomalous  got=anomalous  conf=100%  /tienda1/publico/autenticar.jsp
  ✅ expected=anomalous  got=anomalous  conf=100%  /tienda1/publico/buscar.jsp?query=%3Cscript%3Ealert(1)%3C%2F
  ✅ expected=anomalous  got=anomalous  conf=100%  /tienda1/publico/caracteristicas.jsp?id=../../../etc/passwd

✅ Model ready — run notebooks 1 and 2.
